In [1]:
# ================================================================
# COMPLETE GOOGLE COLAB ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ================================================================
# 1. LOAD CLEANED DATASET
# ================================================================

file_path = "/content/cleaned_sample_dataset.xlsx"

df = pd.read_excel(file_path)

print("=" * 70)
print("TASK 14 - FAIRNESS & BIAS ANALYSIS")
print("=" * 70)

print("\nDataset loaded successfully.")

print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

print("\nColumns:")
print(df.columns.tolist())


# ================================================================
# 2. BASIC DATA OVERVIEW
# ================================================================

print("\n" + "=" * 70)
print("1. BASIC DATA OVERVIEW")
print("=" * 70)

print("\nDataset Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Records:")
display(df.head())

print("\nStatistical Summary:")
display(df.describe(include="all").T)


# ================================================================
# 3. DATA QUALITY VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("2. DATA QUALITY VALIDATION")
print("=" * 70)

print("\nMissing Values:")
missing_report = pd.DataFrame({
    "Missing_Values": df.isnull().sum(),
    "Missing_Percentage":
        (df.isnull().sum() / len(df) * 100).round(2)
})

display(missing_report)

print("\nDuplicate Records:")
print(df.duplicated().sum())


# ================================================================
# 4. IDENTIFY OUTCOME VARIABLE
# ================================================================

print("\n" + "=" * 70)
print("3. OUTCOME VARIABLE")
print("=" * 70)

outcome_column = "decision"

print("Outcome Variable:", outcome_column)

print("\nOutcome Distribution:")
outcome_distribution = df[outcome_column].value_counts()

display(outcome_distribution.to_frame("Count"))

print("\nOutcome Percentage:")
outcome_percentage = (
    df[outcome_column]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Percentage")
)

display(outcome_percentage)


# ================================================================
# 5. CREATE BINARY OUTCOME
# ================================================================

df["approved_flag"] = (
    df["decision"]
    .str.lower()
    .eq("approved")
    .astype(int)
)

print("\nApproved = 1")
print("Rejected = 0")

print("\nOverall Approval Rate:",
      round(df["approved_flag"].mean() * 100, 2), "%")


# ================================================================
# 6. GROUP DISTRIBUTION
# ================================================================

print("\n" + "=" * 70)
print("4. GROUP DISTRIBUTION")
print("=" * 70)

group_columns = [
    "gender",
    "region",
    "education"
]

for col in group_columns:

    print("\n---", col.upper(), "---")

    group_count = (
        df[col]
        .value_counts(dropna=False)
        .to_frame("Count")
    )

    group_count["Percentage"] = (
        group_count["Count"] /
        len(df) * 100
    ).round(2)

    display(group_count)


# ================================================================
# 7. OUTCOME DISPARITY BY GENDER
# ================================================================

print("\n" + "=" * 70)
print("5. OUTCOME DISPARITY - GENDER")
print("=" * 70)

gender_analysis = (
    df.groupby("gender")
    .agg(
        Total_Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

gender_analysis["Approval_Rate"] = (
    gender_analysis["Approval_Rate"] * 100
).round(2)

gender_analysis["Rejection_Rate"] = (
    100 - gender_analysis["Approval_Rate"]
).round(2)

display(gender_analysis)


# ================================================================
# 8. OUTCOME DISPARITY BY REGION
# ================================================================

print("\n" + "=" * 70)
print("6. OUTCOME DISPARITY - REGION")
print("=" * 70)

region_analysis = (
    df.groupby("region")
    .agg(
        Total_Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

region_analysis["Approval_Rate"] = (
    region_analysis["Approval_Rate"] * 100
).round(2)

region_analysis["Rejection_Rate"] = (
    100 - region_analysis["Approval_Rate"]
).round(2)

display(region_analysis)


# ================================================================
# 9. OUTCOME DISPARITY BY EDUCATION
# ================================================================

print("\n" + "=" * 70)
print("7. OUTCOME DISPARITY - EDUCATION")
print("=" * 70)

education_analysis = (
    df.groupby("education")
    .agg(
        Total_Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

education_analysis["Approval_Rate"] = (
    education_analysis["Approval_Rate"] * 100
).round(2)

education_analysis["Rejection_Rate"] = (
    100 - education_analysis["Approval_Rate"]
).round(2)

display(education_analysis)


# ================================================================
# 10. DISPARITY CALCULATION FUNCTION
# ================================================================

def calculate_disparity(data, group_column):

    result = (
        data.groupby(group_column)["approved_flag"]
        .agg(["count", "sum", "mean"])
        .reset_index()
    )

    result.columns = [
        group_column,
        "Applications",
        "Approved",
        "Approval_Rate"
    ]

    result["Approval_Rate"] = (
        result["Approval_Rate"] * 100
    ).round(2)

    highest_rate = result["Approval_Rate"].max()
    lowest_rate = result["Approval_Rate"].min()

    result["Approval_Rate_Difference_From_Highest"] = (
        highest_rate - result["Approval_Rate"]
    ).round(2)

    if highest_rate != 0:
        result["Selection_Rate_Ratio"] = (
            result["Approval_Rate"] /
            highest_rate
        ).round(3)
    else:
        result["Selection_Rate_Ratio"] = np.nan

    return result


# ================================================================
# 11. GENDER DISPARITY METRICS
# ================================================================

print("\n" + "=" * 70)
print("8. GENDER DISPARITY METRICS")
print("=" * 70)

gender_disparity = calculate_disparity(
    df,
    "gender"
)

display(gender_disparity)


# ================================================================
# 12. REGION DISPARITY METRICS
# ================================================================

print("\n" + "=" * 70)
print("9. REGION DISPARITY METRICS")
print("=" * 70)

region_disparity = calculate_disparity(
    df,
    "region"
)

display(region_disparity)


# ================================================================
# 13. EDUCATION DISPARITY METRICS
# ================================================================

print("\n" + "=" * 70)
print("10. EDUCATION DISPARITY METRICS")
print("=" * 70)

education_disparity = calculate_disparity(
    df,
    "education"
)

display(education_disparity)


# ================================================================
# 14. IDENTIFY HIGHEST & LOWEST APPROVAL GROUPS
# ================================================================

print("\n" + "=" * 70)
print("11. HIGHEST & LOWEST APPROVAL GROUPS")
print("=" * 70)

def highest_lowest(result, group_column):

    highest = result.loc[
        result["Approval_Rate"].idxmax()
    ]

    lowest = result.loc[
        result["Approval_Rate"].idxmin()
    ]

    print("\nGroup:", group_column)

    print(
        "Highest Approval:",
        highest[group_column],
        "->",
        highest["Approval_Rate"],
        "%"
    )

    print(
        "Lowest Approval:",
        lowest[group_column],
        "->",
        lowest["Approval_Rate"],
        "%"
    )

    print(
        "Absolute Difference:",
        round(
            highest["Approval_Rate"] -
            lowest["Approval_Rate"],
            2
        ),
        "percentage points"
    )


highest_lowest(
    gender_disparity,
    "gender"
)

highest_lowest(
    region_disparity,
    "region"
)

highest_lowest(
    education_disparity,
    "education"
)


# ================================================================
# 15. FOUR-FIFTHS / 80% RULE SCREENING
# ================================================================

print("\n" + "=" * 70)
print("12. 80% RULE SCREENING")
print("=" * 70)

print("""
This is a screening indicator, not a final legal conclusion.

A selection-rate ratio below 0.80 can indicate a potential
disparity that requires further investigation.
""")

def eighty_percent_screen(result, group_column):

    result = result.copy()

    highest_rate = result["Approval_Rate"].max()

    result["Selection_Rate_Ratio"] = (
        result["Approval_Rate"] /
        highest_rate
    ).round(3)

    result["Potential_Disparity"] = np.where(
        result["Selection_Rate_Ratio"] < 0.80,
        "Review Required",
        "No Automatic Flag"
    )

    print("\n", group_column.upper())

    display(result)

    return result


gender_80 = eighty_percent_screen(
    gender_disparity,
    "gender"
)

region_80 = eighty_percent_screen(
    region_disparity,
    "region"
)

education_80 = eighty_percent_screen(
    education_disparity,
    "education"
)


# ================================================================
# 16. OUTCOME DISPARITY BY GENDER + REGION
# ================================================================

print("\n" + "=" * 70)
print("13. GENDER + REGION ANALYSIS")
print("=" * 70)

gender_region = (
    df.groupby(["gender", "region"])
    .agg(
        Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

gender_region["Approval_Rate"] = (
    gender_region["Approval_Rate"] * 100
).round(2)

display(gender_region)


# ================================================================
# 17. GENDER + EDUCATION ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("14. GENDER + EDUCATION ANALYSIS")
print("=" * 70)

gender_education = (
    df.groupby(["gender", "education"])
    .agg(
        Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

gender_education["Approval_Rate"] = (
    gender_education["Approval_Rate"] * 100
).round(2)

display(gender_education)


# ================================================================
# 18. INCOME ANALYSIS BY GROUP
# ================================================================

print("\n" + "=" * 70)
print("15. INCOME ANALYSIS BY GROUP")
print("=" * 70)

income_by_gender = (
    df.groupby("gender")["annual_income"]
    .agg(
        Average_Income="mean",
        Median_Income="median"
    )
    .round(2)
    .reset_index()
)

display(income_by_gender)


# ================================================================
# 19. CREDIT SCORE ANALYSIS BY GROUP
# ================================================================

print("\n" + "=" * 70)
print("16. CREDIT SCORE ANALYSIS BY GROUP")
print("=" * 70)

credit_by_gender = (
    df.groupby("gender")["credit_score"]
    .agg(
        Average_Credit_Score="mean",
        Median_Credit_Score="median"
    )
    .round(2)
    .reset_index()
)

display(credit_by_gender)


# ================================================================
# 20. APPROVAL RATE VS CREDIT SCORE
# ================================================================

print("\n" + "=" * 70)
print("17. CREDIT SCORE BAND ANALYSIS")
print("=" * 70)

df["credit_score_band"] = pd.cut(
    df["credit_score"],
    bins=[299, 579, 669, 739, 850],
    labels=[
        "Poor",
        "Fair",
        "Good",
        "Very Good"
    ]
)

credit_band_analysis = (
    df.groupby(
        "credit_score_band",
        observed=False
    )
    .agg(
        Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

credit_band_analysis["Approval_Rate"] = (
    credit_band_analysis["Approval_Rate"] * 100
).round(2)

display(credit_band_analysis)


# ================================================================
# 21. APPROVAL RATE VS INCOME BAND
# ================================================================

print("\n" + "=" * 70)
print("18. INCOME BAND ANALYSIS")
print("=" * 70)

df["income_band"] = pd.cut(
    df["annual_income"],
    bins=[0, 40000, 70000, 100000, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

income_band_analysis = (
    df.groupby(
        "income_band",
        observed=False
    )
    .agg(
        Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

income_band_analysis["Approval_Rate"] = (
    income_band_analysis["Approval_Rate"] * 100
).round(2)

display(income_band_analysis)


# ================================================================
# 22. APPROVAL RATE VS AGE GROUP
# ================================================================

print("\n" + "=" * 70)
print("19. AGE BAND ANALYSIS")
print("=" * 70)

df["age_band"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, 100],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56+"
    ]
)

age_analysis = (
    df.groupby(
        "age_band",
        observed=False
    )
    .agg(
        Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

age_analysis["Approval_Rate"] = (
    age_analysis["Approval_Rate"] * 100
).round(2)

display(age_analysis)


# ================================================================
# 23. CONSENT ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("20. CONSENT ANALYSIS")
print("=" * 70)

consent_column = "consent_for_fairness_audit"

consent_analysis = (
    df.groupby(consent_column)
    .agg(
        Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

consent_analysis["Approval_Rate"] = (
    consent_analysis["Approval_Rate"] * 100
).round(2)

display(consent_analysis)

print("\nConsent Distribution:")
display(
    df[consent_column]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Percentage")
)


# ================================================================
# 24. SAMPLE SIZE CHECK
# ================================================================

print("\n" + "=" * 70)
print("21. SAMPLE SIZE CHECK")
print("=" * 70)

for col in group_columns:

    print("\n", col.upper())

    sample_size = (
        df[col]
        .value_counts()
        .to_frame("Sample_Size")
    )

    sample_size["Small_Group_Flag"] = np.where(
        sample_size["Sample_Size"] < 30,
        "Small Sample - Interpret Carefully",
        "Adequate for Descriptive Analysis"
    )

    display(sample_size)


# ================================================================
# 25. OVERALL FAIRNESS SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("22. OVERALL FAIRNESS SUMMARY")
print("=" * 70)

summary_rows = []

for name, result, group_column in [
    ("Gender", gender_disparity, "gender"),
    ("Region", region_disparity, "region"),
    ("Education", education_disparity, "education")
]:

    highest = result["Approval_Rate"].max()
    lowest = result["Approval_Rate"].min()

    summary_rows.append({
        "Group_Type": name,
        "Highest_Approval_Rate": highest,
        "Lowest_Approval_Rate": lowest,
        "Absolute_Disparity":
            round(highest - lowest, 2),
        "Lowest_to_Highest_Ratio":
            round(lowest / highest, 3)
            if highest != 0 else np.nan
    })

fairness_summary = pd.DataFrame(summary_rows)

display(fairness_summary)


# ================================================================
# 26. IDENTIFY AREAS REQUIRING REVIEW
# ================================================================

print("\n" + "=" * 70)
print("23. AREAS REQUIRING REVIEW")
print("=" * 70)

review_items = []

for _, row in fairness_summary.iterrows():

    if row["Lowest_to_Highest_Ratio"] < 0.80:

        review_items.append({
            "Group_Type": row["Group_Type"],
            "Issue":
                "Selection-rate ratio below 0.80",
            "Action":
                "Investigate potential outcome disparity"
        })

    else:

        review_items.append({
            "Group_Type": row["Group_Type"],
            "Issue":
                "No automatic 80% rule flag",
            "Action":
                "Continue monitoring"
        })

review_df = pd.DataFrame(review_items)

display(review_df)


# ================================================================
# 27. MONITORING BASELINE
# ================================================================

print("\n" + "=" * 70)
print("24. FAIRNESS MONITORING BASELINE")
print("=" * 70)

monitoring_baseline = fairness_summary.copy()

monitoring_baseline["Monitoring_Status"] = "Baseline Established"

display(monitoring_baseline)


# ================================================================
# 28. CREATE MONTHLY MONITORING DATA
# ================================================================

print("\n" + "=" * 70)
print("25. MONTHLY FAIRNESS MONITORING")
print("=" * 70)

df["application_month"] = (
    pd.to_datetime(df["application_date"])
    .dt.to_period("M")
    .astype(str)
)

monthly_monitoring = (
    df.groupby(
        ["application_month", "gender"]
    )
    .agg(
        Applications=("approved_flag", "size"),
        Approved=("approved_flag", "sum"),
        Approval_Rate=("approved_flag", "mean")
    )
    .reset_index()
)

monthly_monitoring["Approval_Rate"] = (
    monthly_monitoring["Approval_Rate"] * 100
).round(2)

display(monthly_monitoring)


# ================================================================
# 29. MONTHLY DISPARITY RANGE
# ================================================================

monthly_disparity = (
    monthly_monitoring
    .groupby("application_month")
    .agg(
        Highest_Approval_Rate=("Approval_Rate", "max"),
        Lowest_Approval_Rate=("Approval_Rate", "min")
    )
    .reset_index()
)

monthly_disparity["Disparity"] = (
    monthly_disparity["Highest_Approval_Rate"] -
    monthly_disparity["Lowest_Approval_Rate"]
).round(2)

monthly_disparity["Lowest_to_Highest_Ratio"] = (
    monthly_disparity["Lowest_Approval_Rate"] /
    monthly_disparity["Highest_Approval_Rate"]
).round(3)

display(monthly_disparity)


# ================================================================
# 30. FINAL FAIRNESS FLAGS
# ================================================================

print("\n" + "=" * 70)
print("26. FINAL FAIRNESS FLAGS")
print("=" * 70)

final_flags = fairness_summary.copy()

final_flags["Fairness_Flag"] = np.where(
    final_flags["Lowest_to_Highest_Ratio"] < 0.80,
    "Potential Disparity - Investigate",
    "No Automatic Flag"
)

display(final_flags)


# ================================================================
# 31. METHODOLOGY SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("27. METHODOLOGY")
print("=" * 70)

methodology = {
    "Outcome":
        "Loan application decision (Approved/Rejected)",

    "Primary Metric":
        "Approval / Selection Rate",

    "Disparity Metric":
        "Difference between highest and lowest group approval rates",

    "Ratio Metric":
        "Lowest group approval rate divided by highest group approval rate",

    "Screening Threshold":
        "0.80 selection-rate ratio used as an investigation screening indicator",

    "Groups Analysed":
        "Gender, Region and Education",

    "Additional Factors":
        "Age, Annual Income, Credit Score and Application patterns",

    "Monitoring":
        "Monthly group-level approval rates and disparity ratio",

    "Limitation":
        "Descriptive analysis alone cannot establish causation or prove discrimination"
}

methodology_df = pd.DataFrame(
    list(methodology.items()),
    columns=["Methodology_Item", "Description"]
)

display(methodology_df)


# ================================================================
# 32. LIMITATIONS
# ================================================================

print("\n" + "=" * 70)
print("28. LIMITATIONS")
print("=" * 70)

limitations = [
    "The analysis identifies statistical disparities but does not establish causation.",
    "Small group sizes can make disparity estimates unstable.",
    "Observed differences may be influenced by legitimate business factors.",
    "The 80% rule is used as a screening indicator, not as a definitive legal conclusion.",
    "Historical data may contain existing bias that cannot be identified from outcome rates alone.",
    "Additional model-level analysis would be required if a predictive model is used.",
    "Fairness metrics should be monitored continuously rather than treated as a one-time audit."
]

for i, item in enumerate(limitations, 1):
    print(f"{i}. {item}")


# ================================================================
# 33. ACTION RECOMMENDATIONS
# ================================================================

print("\n" + "=" * 70)
print("29. RECOMMENDED ACTIONS")
print("=" * 70)

actions = [
    "Establish the current results as the fairness baseline.",
    "Investigate groups showing large approval-rate differences.",
    "Check whether legitimate business variables explain observed differences.",
    "Review small-group results carefully before drawing conclusions.",
    "Monitor approval rates and selection-rate ratios regularly.",
    "Document methodology, assumptions, limitations and decisions.",
    "Escalate persistent or material disparities for deeper investigation."
]

for i, action in enumerate(actions, 1):
    print(f"{i}. {action}")


# ================================================================
# 34. EXPORT ALL ANALYSIS RESULTS TO EXCEL
# ================================================================

print("\n" + "=" * 70)
print("30. EXPORTING ANALYSIS RESULTS")
print("=" * 70)

output_file = "/content/task14_fairness_analysis_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="Analysed_Data",
        index=False
    )

    missing_report.to_excel(
        writer,
        sheet_name="Data_Quality"
    )

    outcome_percentage.to_excel(
        writer,
        sheet_name="Outcome_Distribution"
    )

    gender_analysis.to_excel(
        writer,
        sheet_name="Gender_Analysis",
        index=False
    )

    region_analysis.to_excel(
        writer,
        sheet_name="Region_Analysis",
        index=False
    )

    education_analysis.to_excel(
        writer,
        sheet_name="Education_Analysis",
        index=False
    )

    gender_disparity.to_excel(
        writer,
        sheet_name="Gender_Disparity",
        index=False
    )

    region_disparity.to_excel(
        writer,
        sheet_name="Region_Disparity",
        index=False
    )

    education_disparity.to_excel(
        writer,
        sheet_name="Education_Disparity",
        index=False
    )

    gender_region.to_excel(
        writer,
        sheet_name="Gender_Region",
        index=False
    )

    gender_education.to_excel(
        writer,
        sheet_name="Gender_Education",
        index=False
    )

    credit_band_analysis.to_excel(
        writer,
        sheet_name="Credit_Band",
        index=False
    )

    income_band_analysis.to_excel(
        writer,
        sheet_name="Income_Band",
        index=False
    )

    age_analysis.to_excel(
        writer,
        sheet_name="Age_Band",
        index=False
    )

    consent_analysis.to_excel(
        writer,
        sheet_name="Consent_Analysis",
        index=False
    )

    fairness_summary.to_excel(
        writer,
        sheet_name="Fairness_Summary",
        index=False
    )

    review_df.to_excel(
        writer,
        sheet_name="Review_Actions",
        index=False
    )

    monthly_monitoring.to_excel(
        writer,
        sheet_name="Monthly_Monitoring",
        index=False
    )

    monthly_disparity.to_excel(
        writer,
        sheet_name="Monthly_Disparity",
        index=False
    )

    final_flags.to_excel(
        writer,
        sheet_name="Final_Flags",
        index=False
    )

    methodology_df.to_excel(
        writer,
        sheet_name="Methodology",
        index=False
    )


# ================================================================
# 35. EXPORT KEY TABLES AS CSV
# ================================================================

fairness_summary.to_csv(
    "/content/fairness_summary.csv",
    index=False
)

gender_disparity.to_csv(
    "/content/gender_disparity.csv",
    index=False
)

region_disparity.to_csv(
    "/content/region_disparity.csv",
    index=False
)

education_disparity.to_csv(
    "/content/education_disparity.csv",
    index=False
)

monthly_disparity.to_csv(
    "/content/monthly_fairness_monitoring.csv",
    index=False
)


# ================================================================
# 36. FINAL COMPLETION MESSAGE
# ================================================================

print("\n" + "=" * 70)
print("TASK 14 GOOGLE COLAB ANALYSIS COMPLETED")
print("=" * 70)

print("""
Completed:

1. Data quality validation
2. Outcome analysis
3. Group distribution analysis
4. Gender disparity analysis
5. Region disparity analysis
6. Education disparity analysis
7. Selection-rate ratio
8. 80% rule screening
9. Intersectional analysis
10. Income analysis
11. Credit-score analysis
12. Age-band analysis
13. Consent analysis
14. Sample-size assessment
15. Fairness baseline
16. Monthly monitoring
17. Methodology
18. Limitations
19. Recommended actions
20. Excel/CSV result export

Charts and dashboard have intentionally NOT been created.
They will be built later in Tableau.
""")

print("\nMain output file:")
print(output_file)

TASK 14 - FAIRNESS & BIAS ANALYSIS

Dataset loaded successfully.
Rows    : 500
Columns : 13

Columns:
['application_id', 'gender', 'age', 'region', 'education', 'experience_years', 'annual_income', 'credit_score', 'loan_amount', 'application_date', 'decision', 'processing_time_days', 'consent_for_fairness_audit']

1. BASIC DATA OVERVIEW

Dataset Shape:
(500, 13)

Data Types:
application_id                        object
gender                                object
age                                    int64
region                                object
education                             object
experience_years                       int64
annual_income                          int64
credit_score                           int64
loan_amount                            int64
application_date              datetime64[ns]
decision                              object
processing_time_days                 float64
consent_for_fairness_audit            object
dtype: object

First 5 Records:


,application_id,gender,age,region,education,experience_years,annual_income,credit_score,loan_amount,application_date,decision,processing_time_days,consent_for_fairness_audit
0,APP10001,Male,37,West,Master,12,88167,729,14379,2025-08-02,Approved,4.90,Yes
1,APP10002,Female,29,North,Master,8,109733,734,42485,2025-10-18,Approved,6.30,Yes
2,APP10003,Female,53,South,Bachelor,26,36097,621,19569,2025-01-22,Approved,6.00,Yes
3,APP10004,Female,40,North,Master,14,40903,698,23441,2025-07-12,Approved,3.70,Yes
4,APP10005,Male,33,South,Bachelor,13,61286,609,31977,2025-05-30,Rejected,2.90,No



Statistical Summary:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
application_id,500,500,APP10500,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,500,3,Female,243,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,500.00,NaN,NaN,NaN,40.58,21.00,30.00,41.00,51.00,60.00,11.86
region,500,4,North,145,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,500,4,Bachelor,233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experience_years,500.00,NaN,NaN,NaN,16.12,0.00,5.00,16.50,26.00,39.00,11.63
annual_income,500.00,NaN,NaN,NaN,"66,210.59","18,000.00","51,026.50","66,535.00","81,952.00","136,602.00","22,201.42"
credit_score,500.00,NaN,NaN,NaN,695.94,471.00,651.75,698.00,744.25,850.00,75.99
loan_amount,500.00,NaN,NaN,NaN,"25,369.82","5,000.00","17,019.00","25,903.00","32,996.25","58,001.00","11,342.16"
application_date,500,NaN,NaN,NaN,2025-07-07 01:00:28.800000256,2025-01-03 00:00:00,2025-04-05 18:00:00,2025-07-12 12:00:00,2025-10-08 12:00:00,2025-12-31 00:00:00,NaN



2. DATA QUALITY VALIDATION

Missing Values:


...


26. FINAL FAIRNESS FLAGS


,Group_Type,Highest_Approval_Rate,Lowest_Approval_Rate,Absolute_Disparity,Lowest_to_Highest_Ratio,Fairness_Flag
0,Gender,90.99,86.01,4.98,0.94,No Automatic Flag
1,Region,91.54,86.90,4.64,0.95,No Automatic Flag
2,Education,95.83,87.41,8.42,0.91,No Automatic Flag



27. METHODOLOGY


,Methodology_Item,Description
0,Outcome,Loan application decision (Approved/Rejected)
1,Primary Metric,Approval / Selection Rate
2,Disparity Metric,Difference between highest and lowest group ap...
3,Ratio Metric,Lowest group approval rate divided by highest ...
4,Screening Threshold,0.80 selection-rate ratio used as an investiga...
5,Groups Analysed,"Gender, Region and Education"
6,Additional Factors,"Age, Annual Income, Credit Score and Applicati..."
7,Monitoring,Monthly group-level approval rates and dispari...
8,Limitation,Descriptive analysis alone cannot establish ca...



28. LIMITATIONS
1. The analysis identifies statistical disparities but does not establish causation.
2. Small group sizes can make disparity estimates unstable.
3. Observed differences may be influenced by legitimate business factors.
4. The 80% rule is used as a screening indicator, not as a definitive legal conclusion.
5. Historical data may contain existing bias that cannot be identified from outcome rates alone.
6. Additional model-level analysis would be required if a predictive model is used.
7. Fairness metrics should be monitored continuously rather than treated as a one-time audit.

29. RECOMMENDED ACTIONS
1. Establish the current results as the fairness baseline.
2. Investigate groups showing large approval-rate differences.
3. Check whether legitimate business variables explain observed differences.
4. Review small-group results carefully before drawing conclusions.
5. Monitor approval rates and selection-rate ratios regularly.
6. Document methodology, assumptions, limitati